# VHH HEFT predictions — NNLO QCD

This notebook computes **HEFT** predictions for $W^\pm HH$ and $ZHH$ at **LO**
and **NNLO QCD**, for the LHC at 13.6 or 14 TeV.

The cross section is the closed form $\sigma = \mathbf{m}(\kappa)^\top \mathbf{A}$
with PDF+$\alpha_s$ uncertainty $\sqrt{\mathbf{m}^\top C\,\mathbf{m}}$ and a 7-point
scale envelope, using coefficients bundled under `data/HEFT/`.

> **Install once** with `pip install -e ".[notebook]"` (package only), then open this
> notebook from the repo root. Restart the kernel after editing `vhh_predict/`.

> For **SMEFT** Wilson coefficients use [`vhh_prediction_SMEFT.ipynb`](vhh_prediction_SMEFT.ipynb).

## Contents

| § | What it does | Main output |
|---|---|---|
| Setup | Editable install check + imports | — |
| 1 | Process, energy, output flags | `analysis` |
| 2 | Spot check at one $\kappa$ point | results table (+ comparison with simulation if enabled) |
| 3 | Scan one $\kappa$ axis and plot | `scan_data` + PNGs (+ optional `.txt`) |
| 4 | Joint multi-axis grid scan | one `results/points/*_x_*.txt` |
| 5 | HEFT benchmark tables | display + `results/tables/wilson_tables.tex` |

See [README.md](README.md) for an overview; `AGENTS.md` for the package API.


## Setup

Requires an editable install from the repo root (defined in `pyproject.toml`):

```bash
pip install -e ".[notebook]"
```

Then `import vhh_predict` works — notebooks do **not** modify `sys.path`.
Restart the kernel after changing package code.


In [ ]:
from vhh_predict.analysis import package_root, plots_dir, tables_dir

REPO_ROOT = package_root()
if not (REPO_ROOT / "data" / "HEFT").is_dir() or not (REPO_ROOT / "vhh_predict").is_dir():
    raise RuntimeError(
        f"Expected 'data/HEFT/' and 'vhh_predict/' under {REPO_ROOT}.\n"
        "Install from the repo root:  pip install -e \".[notebook]\"\n"
        "Then start Jupyter from that same directory."
    )

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from vhh_predict import (
    CHANNELS,
    TABLE_ENERGIES_TEV,
    WILSON_INTERVALS,
    build_channel_tables,
    latex_wilson_tables_for_process,
    load_analysis,
    resolve_scan_axis,
    scan_and_save,
    scan_axes,
    scan_grid_and_save,
    sm_kappa,
)
from vhh_predict.core import spot_check_caption, spot_check_table
from vhh_predict.plot_style import (
    default_plot_title,
    plot_style_with_layout,
    scan_plot_filename_stem,
)
from vhh_predict.plots import (
    plot_sigma_nnlo_and_enhancement_nnlo,
    plot_sigma_nnlo_and_kfactor,
)
from vhh_predict.tables import KAPPA_PLAIN, ZHH_TABLE_GROUPS

PLOTS_DIR = plots_dir()
TABLES_DIR = tables_dir()
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root: {REPO_ROOT}")


## 1. Configuration

| Variable | Meaning |
|----------|---------|
| `PROCESS` | `"WplusHH"`, `"WminusHH"`, or `"ZHH"` |
| `ENERGY_TEV` | `13.6` or `14.0` |
| `COMPARE_SIMULATION` | Compare to bundled HEFT `.out` files in §2 only |
| `UNCERTAINTIES_AS_PERCENT` | Print uncertainties as % (`False` → fb) |
| `SAVE_SCAN_POINTS` | Write §3 / §4 scan tables to `results/points/` |
| `SAVE_PLOTS` | Write §3 plot PNGs to `results/plots/` |
| `SIGMA_INSET`, `LEGEND_LOC`, `INSET_LOC` | Plot layout (§3) |

Wilson coefficients $\kappa$. SM point: all $\kappa = 1$ (`sm_kappa(PROCESS)`).

Use the **Key** in `KAPPA` / scans / `predict`, e.g. `scan_axis = "kappa_t"`
(`WILSON_INTERVALS` in `vhh_predict/tables.py`).

### $W^\pm HH$ — $\kappa$ tuple $(\kappa_\lambda,\, \kappa_W,\, \kappa_{2W})$

| Coefficient | Key | Interval (95% CL) |
|-------------|-----|-------------------|
| $\kappa_\lambda$ | `kappa_lambda` | [−0.7, 6.1] |
| $\kappa_W$ | `kappa_w` | [0.8, 1.2] |
| $\kappa_{2W}$ | `kappa_2w` | [0.7, 1.3] |

### $ZHH$ — $\kappa$ tuple $(\kappa_\lambda,\, \kappa_Z,\, \kappa_{2Z},\, \kappa_t)$

| Coefficient | Key | Interval (95% CL) |
|-------------|-----|-------------------|
| $\kappa_\lambda$ | `kappa_lambda` | [−0.7, 6.1] |
| $\kappa_Z$ | `kappa_z` | [0.9, 1.2] |
| $\kappa_{2Z}$ | `kappa_2z` | [0.7, 1.3] |
| $\kappa_t$ | `kappa_t` | [0.8, 1.2] |

> Scan axes are all process $\kappa$ keys (`scan_axes(PROCESS)`), printed in the next cell.


In [ ]:
PROCESS = "ZHH"
ENERGY_TEV = 14.0

COMPARE_SIMULATION = True
UNCERTAINTIES_AS_PERCENT = True
SAVE_SCAN_POINTS = True
SAVE_PLOTS = True

SIGMA_INSET = False
LEGEND_LOC = "lower right"
INSET_LOC = "upper left"

PLOT_STYLE = plot_style_with_layout(
    legend_loc=LEGEND_LOC,
    inset_loc=INSET_LOC,
    sigma_inset=SIGMA_INSET,
)

analysis = load_analysis(PROCESS, ENERGY_TEV)
nn_label = analysis.nnlo_label

print(f"{PROCESS} @ {ENERGY_TEV} TeV  |  scan axes: {', '.join(scan_axes(PROCESS))}")


## 2. Spot check

Set **`KAPPA`** — the Wilson-coefficient point for this table only.

| Process | `KAPPA` tuple | SM |
|---------|---------------|-----|
| `WplusHH`, `WminusHH` | `(κ_λ, κ_W, κ_{2W})` — 3 numbers | all `1.0` |
| `ZHH` | `(κ_λ, κ_Z, κ_{2Z}, κ_t)` — 4 numbers | all `1.0` |

The table shows **$\sigma_{\mathrm{LO}}$**, **$\sigma_{\mathrm{NNLO}}$**, and the **$K$-factor**, with scale / PDF+$\alpha_s$ uncertainties.
When `COMPARE_SIMULATION=True`, matching `.out` files under `data/HEFT/.../Simulation/` are compared if available.


In [ ]:
KAPPA = sm_kappa(PROCESS)  # SM: all 1.0; e.g. ZHH (3.0, 1.0, 1.0, 1.0) → κ_λ = 3

display(Markdown(f"### {spot_check_caption(analysis, KAPPA)}"))
display(
    spot_check_table(
        analysis,
        KAPPA,
        as_percent=UNCERTAINTIES_AS_PERCENT,
        compare_simulation=COMPARE_SIMULATION,
    )
)


## 3. Scan and plots

Vary **one** Wilson coefficient; all others stay at **SM** ($\kappa = 1$).

- `scan_axis` — one of the keys printed in §1 (`kappa_lambda`, `kappa_t`, …)
- `scan_vmin`, `scan_vmax` — scan window (defaults below use the 95% CL interval)
- `scan_n_points` — grid resolution


In [ ]:
scan_axis = "kappa_t"
wilson_lo, wilson_hi = WILSON_INTERVALS[scan_axis]
scan_vmin, scan_vmax = wilson_lo, wilson_hi
scan_n_points = 400

print(f"Interval for {KAPPA_PLAIN[scan_axis]}: [{wilson_lo:g}, {wilson_hi:g}]")
print(f"Scan / plot window:                  [{scan_vmin:g}, {scan_vmax:g}]")

scan_data, scan_points_file = scan_and_save(
    analysis,
    scan_axis,
    vmin=scan_vmin,
    vmax=scan_vmax,
    n_points=scan_n_points,
    uncertainties=True,
    save=SAVE_SCAN_POINTS,
)
if SAVE_SCAN_POINTS:
    print(f"Saved {scan_points_file}")

_, scan_x_key = resolve_scan_axis(PROCESS, scan_axis)
plot_title = default_plot_title(PROCESS, ENERGY_TEV, nn_label)
prefix = scan_plot_filename_stem(PROCESS, ENERGY_TEV, scan_x_key, scan_vmin, scan_vmax)

_nnlo_kw = dict(
    style=PLOT_STYLE,
    nnlo_label=nn_label,
    xmin=scan_vmin,
    xmax=scan_vmax,
    save=SAVE_PLOTS,
)

plot_sigma_nnlo_and_kfactor(
    scan_data,
    title=plot_title,
    sigma_inset=SIGMA_INSET,
    output=PLOTS_DIR / f"{prefix}_sigma_nnlo_K.png",
    **_nnlo_kw,
)
plt.show()

plot_sigma_nnlo_and_enhancement_nnlo(
    scan_data,
    title=plot_title,
    sigma_inset=SIGMA_INSET,
    show_enhancement_uncertainty=False,
    output=PLOTS_DIR / f"{prefix}_sigma_nnlo_sigmaSM_{nn_label}.png",
    **_nnlo_kw,
)
plt.show()


## 4. Joint multi-axis scan (Cartesian grid)

Vary **all** listed $\kappa$ axes **at once** (Cartesian product) and write **one** `.txt` under `results/points/`.
Non-scanned $\kappa$ stay at **SM** ($= 1$).

- `BATCH_SCAN_AXES` — axes to vary jointly
- `BATCH_SCAN_WINDOWS` — optional `{axis: (vmin, vmax)}`; omitted axes use `WILSON_INTERVALS`
- `BATCH_SCAN_N_POINTS` — points **per axis** (2 × 40 → 1600 rows)

Set `BATCH_SCAN_AXES = ()` to skip.


In [ ]:
BATCH_SCAN_AXES = (
    "kappa_lambda",
    "kappa_z",
)
BATCH_SCAN_N_POINTS = 40  # per axis → 40×40 = 1600 points for 2 axes
BATCH_SCAN_WINDOWS = {}

if BATCH_SCAN_AXES:
    scan_data, path = scan_grid_and_save(
        analysis,
        BATCH_SCAN_AXES,
        windows=BATCH_SCAN_WINDOWS or None,
        n_points=BATCH_SCAN_N_POINTS,
        save=SAVE_SCAN_POINTS,
        uncertainties=False,
    )
    n_total = len(next(iter(scan_data.values())))
    print(f"Joint grid scan: {len(BATCH_SCAN_AXES)} axes × {BATCH_SCAN_N_POINTS} pts → {n_total} points")
    for axis in BATCH_SCAN_AXES:
        x_key = resolve_scan_axis(PROCESS, axis)[1]
        lo, hi = (BATCH_SCAN_WINDOWS or {}).get(axis) or WILSON_INTERVALS[x_key]
        print(f"  {KAPPA_PLAIN[x_key]}  [{lo:g}, {hi:g}]")
    print(f"  →  {path if SAVE_SCAN_POINTS else '(not saved)'}")
else:
    print("Joint scan skipped (BATCH_SCAN_AXES is empty).")


## 5. HEFT benchmark tables

σ tables cover **every** scan-axis $\kappa$ (`scan_axes(PROCESS)`):

| Channel | Groups (display only) | Keys |
|---------|----------------------|------|
| `WplusHH`, `WminusHH` | one table | `kappa_lambda`, `kappa_w`, `kappa_2w` |
| `ZHH` | group 1 | `kappa_lambda` ($\kappa_\lambda$), `kappa_t` ($\kappa_t$) |
| `ZHH` | group 2 | `kappa_z` ($\kappa_Z$), `kappa_2z` ($\kappa_{2Z}$) |

For each selected $\kappa$, σ is evaluated at SM and at that coefficient’s interval **min/max** (others held at SM $= 1$), for both 13.6 and 14 TeV. ZHH prints **one table per group**.

The LaTeX file `results/tables/wilson_tables.tex` collects the same benchmark tables.

Grouping for ZHH is `ZHH_TABLE_GROUPS` in `vhh_predict/tables.py`; axes themselves come from `scan_axes`.


In [ ]:
latex_path = TABLES_DIR / "wilson_tables.tex"

print("σ benchmark κ (all scan axes):")
for process in CHANNELS:
    axes = scan_axes(process)
    print(f"  {process}: {', '.join(axes)}")
for i, group in enumerate(ZHH_TABLE_GROUPS, start=1):
    print(f"  ZHH group {i}: keys={list(group)}")
    print(f"               symbols={', '.join(KAPPA_PLAIN[k] for k in group)}")

latex_blocks = []
for process in CHANNELS:
    display(Markdown(f"## {process}"))
    tables = build_channel_tables(process, energies_tev=TABLE_ENERGIES_TEV)
    if process == "ZHH":
        for axes in ZHH_TABLE_GROUPS:
            key = "_".join(axes)
            title = (
                f"({', '.join(axes)}) — "
                + ", ".join(KAPPA_PLAIN[a] for a in axes)
            )
            display(Markdown(f"### {title}"))
            display(tables[key])
    else:
        display(Markdown(f"### all scan axes ({', '.join(scan_axes(process))})"))
        display(tables)
    tex = latex_wilson_tables_for_process(process, energies_tev=TABLE_ENERGIES_TEV)
    latex_blocks.append(tex)

latex_path.write_text("\n\n".join(latex_blocks), encoding="utf-8")
print(f"Saved {latex_path}")
